In [3]:
import pandas as pd
import numpy as np

In [4]:
# 1. Datasets load karein (Ensure clean_shootings.csv aur clean_agencies.csv uploaded hain)
shootings = pd.read_csv('../Data/Cleaned/clean_shootings.csv')
agencies = pd.read_csv('../Data/Cleaned/clean_agencies.csv')

In [5]:
# 2. Merge Data
df = pd.merge(shootings, agencies, left_on='agency_id', right_on='id', how='left', suffixes=('_shoot', '_agency'))

In [6]:
# KPI 1: Age Groups (Perfect for Tableau Bar Charts / Demographic Pyramids)
bins = [0, 17, 24, 34, 44, 54, 150]
labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55+']
df['Age_Group'] = pd.cut(df['age'], bins=bins, labels=labels)

In [7]:
# KPI 2: Is Unarmed Flag (Crucial binary filter for dashboards)
df['Is_Unarmed'] = df['armed_with'].apply(lambda x: True if str(x).strip().lower() == 'unarmed' else False)

In [8]:
# KPI 3: Time Intelligence (Extracting explicitly makes Tableau filtering easier)
df['date'] = pd.to_datetime(df['date'])
df['Year'] = df['date'].dt.year
df['Month'] = df['date'].dt.month_name()
df['Day_of_Week'] = df['date'].dt.day_name()

In [9]:
# KPI 4: Fleeing Status Grouping (Simplifies multiple "fleeing" types into a binary + unknown)
df['Flee_Group'] = df['flee_status'].fillna('Unknown')
df['Flee_Group'] = df['Flee_Group'].apply(lambda x: 'Not Fleeing' if x == 'not' else ('Unknown' if x == 'Unknown' else 'Fleeing'))

In [10]:
# KPI 5: Agency Shooting Volume (Segment agencies by historical footprint)
def categorize_agency(total):
    if pd.isna(total): return 'Unknown'
    if total <= 5: return 'Low Vol (1-5)'
    elif total <= 20: return 'Medium Vol (6-20)'
    else: return 'High Vol (>20)'
df['Agency_Shooting_Volume'] = df['total_shootings'].apply(categorize_agency)

In [11]:
# 4. CLEAN UP AND RENAME COLUMNS FOR TABLEAU EASE OF USE
# ---------------------------------------------------------
df = df.rename(columns={
    'id_shoot': 'Incident_ID',
    'name_shoot': 'Victim_Name',
    'state_shoot': 'Incident_State',
    'name_agency': 'Agency_Name',
    'state_agency': 'Agency_State',
    'type': 'Agency_Type'
})

In [12]:
# Drop duplicate IDs to keep the dataset clean
if 'id_agency' in df.columns:
    df = df.drop(columns=['id_agency'])

In [14]:
# ---------------------------------------------------------
# 5. COLAB FILES SECTION ME SAVE KAREIN
# ---------------------------------------------------------
output_filename = '../Data/Final/final_data.csv'
df.to_csv(output_filename, index=False)

print(f"Success! Data clean ho gaya hai aur '{output_filename}' Colab ke Files section me save ho gayi hai.")
print(f"Total Rows: {len(df)}, Total Columns: {len(df.columns)}")

Success! Data clean ho gaya hai aur '../Data/Final/final_data.csv' Colab ke Files section me save ho gayi hai.
Total Rows: 11265, Total Columns: 29
